# Notebook 1: Data Preparation & EDA
Thực thi trên Kaggle: Cài đặt các thư viện cần thiết bằng lệnh dưới
!pip install datasets pandas matplotlib seaborn transformers

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from transformers import AutoTokenizer

## 1. Tải dữ liệu và Lấy mẫu (Sampling)
Dùng bộ `nam194/vietnews`

In [ ]:
# Tải dataset
print("Loading dataset...")
dataset = load_dataset("nam194/vietnews", split="train")

# Shuffle và lấy subset cố định (để tái hiện kết quả, fairness)
dataset = dataset.shuffle(seed=42)

# Lấy 12,000 mẫu đầu tiên để chia 10k/1k/1k
subset = dataset.select(range(12000))

# Chia tập train/val/test
train_dataset = subset.select(range(0, 10000))
val_dataset = subset.select(range(10000, 11000))
test_dataset = subset.select(range(11000, 12000))

print(f"Train size: {len(train_dataset)}")
print(f"Val size: {len(val_dataset)}")
print(f"Test size: {len(test_dataset)}")

## 2. EDA (Exploratory Data Analysis)
Đo đạc phân bố chiều dài token sử dụng tokenizer của BARTpho-syllable.
LƯU Ý: Ta sử dụng `vinai/bartpho-syllable` thay vì bản word-level 
để tránh phải cài đặt `VnCoreNLP` phức tạp trên Kaggle, model vẫn đảm bảo độ chính xác cực tốt.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("vinai/bartpho-syllable")

def get_lengths(example):
    return {
        "article_len": len(tokenizer(example["article"], truncation=False)["input_ids"]),
        "summary_len": len(tokenizer(example["abstract"], truncation=False)["input_ids"])
    }

print("Đang tính toán chiều dài token...")
eda_dataset = train_dataset.map(get_lengths, num_proc=4)

# Chuyển sang pandas để vẽ biểu đồ
df = eda_dataset.to_pandas()

In [ ]:
# Vẽ biểu đồ phân bố
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
sns.histplot(df["article_len"], bins=50, color='blue', kde=True)
plt.title("Phân bố chiều dài Article (Tokens)")
plt.xlabel("Số lượng tokens")

plt.subplot(1, 2, 2)
sns.histplot(df["summary_len"], bins=50, color='orange', kde=True)
plt.title("Phân bố chiều dài Summary (Tokens)")
plt.xlabel("Số lượng tokens")

plt.tight_layout()
plt.show()

## 3. Lưu dữ liệu
Lưu ra file CSV để sử dụng ở các Notebook sau.
KAGGLE TIP: Tạo Kaggle Dataset từ thư mục output này để import vào các Notebook sau.

In [ ]:
train_dataset.to_csv("train_10k.csv", index=False)
val_dataset.to_csv("val_1k.csv", index=False)
test_dataset.to_csv("test_1k.csv", index=False)
print("Đã lưu các file train_10k.csv, val_1k.csv, test_1k.csv thành công!")